In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Load Breast Cancer Wisconsin Diagnostic Dataset
data = load_breast_cancer()
X = data.data.astype(float)

# sklearn: 0 = malignant, 1 = benign
# Required encoding: benign = 0, malignant = 1
y = 1 - data.target

feature_names = list(data.feature_names)
class_names = {0: "Benign", 1: "Malignant"}

# Split and normalize
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# Sigmoid function
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# BPNN with sigmoid activation and backpropagation
class BPNN:
    def __init__(self, hidden1=16, hidden2=8, learning_rate=0.01, epochs=300):
        self.hidden1 = hidden1
        self.hidden2 = hidden2
        self.lr = learning_rate
        self.max_epochs = epochs
        self.weights = None
        self.biases = None
        self.epochs = 0
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        n, features = X.shape

        rng = np.random.default_rng(42)
        self.weights = [
            rng.normal(0, 0.1, (features, self.hidden1)),
            rng.normal(0, 0.1, (self.hidden1, self.hidden2)),
            rng.normal(0, 0.1, (self.hidden2, 1))
        ]
        self.biases = [
            np.zeros((1, self.hidden1)),
            np.zeros((1, self.hidden2)),
            np.zeros((1, 1))
        ]

        for epoch in range(self.max_epochs):
            # Forward propagation
            z1 = X @ self.weights[0] + self.biases[0]
            a1 = sigmoid(z1)

            z2 = a1 @ self.weights[1] + self.biases[1]
            a2 = sigmoid(z2)

            z3 = a2 @ self.weights[2] + self.biases[2]
            output = sigmoid(z3)

            # Binary cross-entropy loss
            eps = 1e-8
            loss = -np.mean(
                y * np.log(output + eps) +
                (1 - y) * np.log(1 - output + eps)
            )
            self.loss_history.append(loss)

            # Backpropagation
            delta3 = (output - y) / n
            grad_w3 = a2.T @ delta3
            grad_b3 = np.sum(delta3, axis=0, keepdims=True)

            delta2 = (delta3 @ self.weights[2].T) * a2 * (1 - a2)
            grad_w2 = a1.T @ delta2
            grad_b2 = np.sum(delta2, axis=0, keepdims=True)

            delta1 = (delta2 @ self.weights[1].T) * a1 * (1 - a1)
            grad_w1 = X.T @ delta1
            grad_b1 = np.sum(delta1, axis=0, keepdims=True)

            self.weights[0] -= self.lr * grad_w1
            self.biases[0] -= self.lr * grad_b1
            self.weights[1] -= self.lr * grad_w2
            self.biases[1] -= self.lr * grad_b2
            self.weights[2] -= self.lr * grad_w3
            self.biases[2] -= self.lr * grad_b3

            self.epochs = epoch + 1

        return self

    def predict_proba(self, X):
        a1 = sigmoid(X @ self.weights[0] + self.biases[0])
        a2 = sigmoid(a1 @ self.weights[1] + self.biases[1])
        return sigmoid(a2 @ self.weights[2] + self.biases[2]).ravel()

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def accuracy(self, X, y):
        return accuracy_score(y, self.predict(X)) * 100

# Single-Hidden-Layer Neural Network
class SHLNN:
    def __init__(self, hidden=16, learning_rate=0.01, epochs=300):
        self.hidden = hidden
        self.lr = learning_rate
        self.max_epochs = epochs
        self.weights = None
        self.biases = None
        self.epochs = 0
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        n, features = X.shape

        rng = np.random.default_rng(123)
        self.weights = [
            rng.normal(0, 0.1, (features, self.hidden)),
            rng.normal(0, 0.1, (self.hidden, 1))
        ]
        self.biases = [
            np.zeros((1, self.hidden)),
            np.zeros((1, 1))
        ]

        for epoch in range(self.max_epochs):
            a1 = sigmoid(X @ self.weights[0] + self.biases[0])
            output = sigmoid(a1 @ self.weights[1] + self.biases[1])

            eps = 1e-8
            loss = -np.mean(
                y * np.log(output + eps) +
                (1 - y) * np.log(1 - output + eps)
            )
            self.loss_history.append(loss)

            delta2 = (output - y) / n
            grad_w2 = a1.T @ delta2
            grad_b2 = np.sum(delta2, axis=0, keepdims=True)

            delta1 = (delta2 @ self.weights[1].T) * a1 * (1 - a1)
            grad_w1 = X.T @ delta1
            grad_b1 = np.sum(delta1, axis=0, keepdims=True)

            self.weights[0] -= self.lr * grad_w1
            self.biases[0] -= self.lr * grad_b1
            self.weights[1] -= self.lr * grad_w2
            self.biases[1] -= self.lr * grad_b2

            self.epochs = epoch + 1

        return self

    def hidden_activations(self, X):
        return sigmoid(X @ self.weights[0] + self.biases[0])

    def predict_proba(self, X):
        a1 = self.hidden_activations(X)
        return sigmoid(a1 @ self.weights[1] + self.biases[1]).ravel()

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def accuracy(self, X, y):
        return accuracy_score(y, self.predict(X)) * 100

# Train models
print("Training neural networks...")
bpnn = BPNN().fit(X_train, y_train)

mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation="logistic",
    solver="adam",
    max_iter=1000,
    random_state=42
)
mlp.fit(X_train, y_train)

shlnn = SHLNN().fit(X_train, y_train)

def confidence_category(output):
    distance = abs(output - 0.5)
    if distance < 0.10:
        return "Low Confidence"
    elif distance < 0.25:
        return "Moderate Confidence"
    return "High Confidence"

# 1. Tumor Classification using BPNN
def tumor_classification():
    outputs = bpnn.predict_proba(X_test)
    predictions = (outputs >= 0.5).astype(int)

    print("\nBPNN TUMOR CLASSIFICATION")
    for i, (actual, pred, output) in enumerate(
        zip(y_test, predictions, outputs), 1
    ):
        print(
            f"Sample {i:3d} | Actual: {class_names[actual]:10s} | "
            f"Predicted: {class_names[pred]:10s} | Output: {output:.4f}"
        )

    print(f"\nAccuracy: {bpnn.accuracy(X_test, y_test):.2f}%")
    print("Epochs:", bpnn.epochs)
    print("Final Loss:", round(bpnn.loss_history[-1], 6))

# 2. Prediction Confidence Analyzer using MLP
def confidence_analyzer():
    outputs = mlp.predict_proba(X_test)[:, list(mlp.classes_).index(1)]
    predictions = mlp.predict(X_test)

    print("\nMLP PREDICTION CONFIDENCE")
    for i, (output, pred) in enumerate(zip(outputs, predictions), 1):
        print(
            f"Sample {i:3d} | Output: {output:.4f} | "
            f"Predicted: {class_names[pred]:10s} | "
            f"{confidence_category(output)}"
        )

# 3. Feature Contribution Analyzer
def feature_contribution_analyzer():
    sample_no = int(input(f"Enter test sample number (1-{len(X_test)}): "))

    if not 1 <= sample_no <= len(X_test):
        print("Invalid sample number.")
        return

    sample = X_test[sample_no - 1].copy()
    baseline = bpnn.predict_proba(sample.reshape(1, -1))[0]
    contributions = []

    for i, name in enumerate(feature_names):
        modified = sample.copy()
        modified[i] = 0.0
        changed_output = bpnn.predict_proba(modified.reshape(1, -1))[0]
        contributions.append((name, abs(baseline - changed_output)))

    contributions.sort(key=lambda x: x[1], reverse=True)

    print("\nFEATURE CONTRIBUTION RANKING")
    for rank, (name, change) in enumerate(contributions, 1):
        print(f"{rank:2d}. {name}: Output change = {change:.6f}")

    print("\nTOP FIVE FEATURES")
    for name, change in contributions[:5]:
        print(f"{name}: {change:.6f}")

# 4. Hidden Neuron Response Analyzer
def hidden_neuron_analyzer():
    activations = shlnn.hidden_activations(X_test)
    columns = [f"Neuron_{i+1}" for i in range(activations.shape[1])]

    activation_df = pd.DataFrame(activations, columns=columns)
    activation_df.insert(0, "Actual Class", [
        class_names[y] for y in y_test
    ])
    activation_df.insert(0, "Sample", np.arange(1, len(X_test) + 1))

    print("\nHIDDEN NEURON ACTIVATION MATRIX")
    print(activation_df.round(4).to_string(index=False))

    benign = activations[y_test == 0].mean(axis=0)
    malignant = activations[y_test == 1].mean(axis=0)

    print("\nHighest Average Activation for Benign:")
    print(f"Neuron {np.argmax(benign)+1}: {np.max(benign):.4f}")

    print("\nHighest Average Activation for Malignant:")
    print(f"Neuron {np.argmax(malignant)+1}: {np.max(malignant):.4f}")

# 5. Input Feature Masking Challenge
def feature_masking_challenge():
    baseline = bpnn.accuracy(X_test, y_test)
    results = []

    for i, name in enumerate(feature_names):
        masked = X_test.copy()
        masked[:, i] = 0.0
        accuracy = bpnn.accuracy(masked, y_test)

        results.append({
            "Masked Feature": name,
            "Masked Accuracy (%)": accuracy,
            "Accuracy Reduction (%)": baseline - accuracy
        })

    df = pd.DataFrame(results).sort_values(
        "Accuracy Reduction (%)", ascending=False
    )

    print("\nFEATURE MASKING RESULTS")
    print(f"Baseline Accuracy: {baseline:.2f}%")
    print(df.round(4).to_string(index=False))
    print("\nMaximum performance drop after masking:",
          df.iloc[0]["Masked Feature"])

# 6. Misclassification Confidence Analyzer
def misclassification_analyzer():
    outputs = bpnn.predict_proba(X_test)
    predictions = (outputs >= 0.5).astype(int)
    incorrect = np.where(predictions != y_test)[0]

    print("\nMISCLASSIFICATION CONFIDENCE ANALYZER")

    if len(incorrect) == 0:
        print("No misclassified samples.")
        return

    records = []

    for i in incorrect:
        records.append({
            "Sample": i + 1,
            "Actual": class_names[y_test[i]],
            "Predicted": class_names[predictions[i]],
            "Sigmoid Output": round(outputs[i], 4),
            "Confidence": confidence_category(outputs[i]),
            "Distance from Threshold": round(abs(outputs[i] - 0.5), 4),
            "Confidently Wrong": abs(outputs[i] - 0.5) >= 0.25
        })

    df = pd.DataFrame(records)
    print(df.to_string(index=False))

    confident_wrong = df[df["Confidently Wrong"]]
    print("\nConfidently Wrong Predictions:", len(confident_wrong))

# 7. BPNN-MLP-SHLNN Comparative Analysis
def comparative_analysis():
    models = [
        ("BPNN", bpnn, bpnn.predict_proba(X_test)),
        ("MLP", mlp, mlp.predict_proba(X_test)[:, list(mlp.classes_).index(1)]),
        ("SHLNN", shlnn, shlnn.predict_proba(X_test))
    ]

    results = []

    for name, net, outputs in models:
        predictions = (outputs >= 0.5).astype(int)
        loss = getattr(net, "loss_history", None)

        results.append({
            "Model": name,
            "Accuracy (%)": accuracy_score(y_test, predictions) * 100,
            "Misclassified": int(np.sum(predictions != y_test)),
            "Final Training Loss": (
                loss[-1] if loss is not None
                else getattr(net, "loss_", np.nan)
            ),
            "Epochs/Iterations": getattr(
                net, "epochs", getattr(net, "n_iter_", "N/A")
            )
        })

    df = pd.DataFrame(results).sort_values(
        "Accuracy (%)", ascending=False
    )

    print("\nBPNN-MLP-SHLNN COMPARATIVE ANALYSIS")
    print(df.round(4).to_string(index=False))
    print("\nBest model by test accuracy:", df.iloc[0]["Model"])

# Menu
def main_menu():
    while True:
        print("\n" + "=" * 65)
        print(" BREAST CANCER BPNN PREDICTION AND FEATURE ANALYSIS")
        print("=" * 65)
        print("1. Tumor Classification using BPNN")
        print("2. Prediction Confidence Analyzer")
        print("3. Feature Contribution Analyzer")
        print("4. Hidden Neuron Response Analyzer")
        print("5. Input Feature Masking Challenge")
        print("6. Misclassification Confidence Analyzer")
        print("7. BPNN-MLP-SHLNN Comparative Analysis")
        print("8. Exit")

        choice = input("\nEnter your choice (1-8): ").strip()

        if choice == "1":
            tumor_classification()
        elif choice == "2":
            confidence_analyzer()
        elif choice == "3":
            feature_contribution_analyzer()
        elif choice == "4":
            hidden_neuron_analyzer()
        elif choice == "5":
            feature_masking_challenge()
        elif choice == "6":
            misclassification_analyzer()
        elif choice == "7":
            comparative_analysis()
        elif choice == "8":
            print("\nPROGRAM FINISHED")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 8.")

main_menu()

Training neural networks...

 BREAST CANCER BPNN PREDICTION AND FEATURE ANALYSIS
1. Tumor Classification using BPNN
2. Prediction Confidence Analyzer
3. Feature Contribution Analyzer
4. Hidden Neuron Response Analyzer
5. Input Feature Masking Challenge
6. Misclassification Confidence Analyzer
7. BPNN-MLP-SHLNN Comparative Analysis
8. Exit
